# Notebook Gold- Implementation of dim tables

In [31]:
import pandas as pd 
import sys
from pathlib import Path




# Ajoute la racine du projet au path Python
sys.path.insert(0, str(Path().resolve().parent))

In [32]:
from src.utils.db import load_table,connect_to_db

## implemantation of dim_date

In [39]:
dim_date=pd.DataFrame({'year':range(2016,2026)})

In [40]:
dim_date

,year
0,2016
1,2017
2,2018
3,2019
4,2020
5,2021
6,2022
7,2023
8,2024
9,2025


In [41]:
dim_date['decade']= (dim_date['year'] // 10) * 10

In [42]:
dim_date.head()

,year,decade
0,2016,2010
1,2017,2010
2,2018,2010
3,2019,2010
4,2020,2020


### schema, envoi et chargement de la table dim_date

In [43]:
from pandas.io.sql import get_schema
engine=connect_to_db()
print(get_schema(dim_date, name='dim_date', schema='gold',keys=['year'],con=engine))

parameters of DATABASE sucessfuly loaded 
connection sucessful

CREATE TABLE gold.dim_date (
	year BIGSERIAL NOT NULL, 
	decade BIGINT, 
	CONSTRAINT dim_date_pk PRIMARY KEY (year)
)




In [38]:
from sqlalchemy import text

ddl = get_schema(dim_date, name='dim_date', schema='gold', keys=['year'],con=engine)

with engine.connect() as conn:
    conn.execute(text(ddl))
    conn.commit()

In [44]:
load_table(dim_date,'dim_date','gold',engine)

load of gold.dim_date done


## Implementation of dim_pollutant

In [77]:
query_f1_4 = """
    SELECT * FROM silver.f1_emission
"""
#use of general function in main
f1_emission= pd.read_sql(query_f1_4, con=engine)

In [78]:
f1_emission['pollutant'].unique()

<StringArray>
[                                 'Carbon dioxide (CO2)',
                            'Hydro-fluorocarbons (HFCS)',
                                         'Ammonia (NH3)',
                                 'Nitrogen oxides (NOX)',
                        'Chromium and compounds (as Cr)',
                         'Cadmium and compounds (as Cd)',
                                  'Sulphur oxides (SOX)',
        'Non-methane volatile organic compounds (NMVOC)',
                            'Lead and compounds (as Pb)',
                                   'Nitrous oxide (N2O)',
                             'Particulate matter (PM10)',
                         'Mercury and compounds (as Hg)',
             'Chlorine and inorganic compounds (as HCl)',
                                               'Benzene',
 '1,1,2,2-tetrachloroethane (TETRACHLOROETHANE-1,1,2,2)',
                                  'Benzo(g,h,i)perylene',
                                           'Naphthalene',


In [79]:
dim_pollutant = f1_emission[['pollutant']].drop_duplicates().reset_index(drop=True)
dim_pollutant['pollutant_id'] = dim_pollutant.index + 1

In [80]:
dim_pollutant['pollutant_id'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46])

In [144]:
mapping_pollutant = dim_pollutant.set_index('pollutant')['pollutant_id'].to_dict()

In [146]:
len(mapping_pollutant)

46

In [81]:
print(get_schema(dim_pollutant, name='dim_pollutant', schema='gold',keys=['pollutant_id'],con=engine))


CREATE TABLE gold.dim_pollutant (
	pollutant TEXT, 
	pollutant_id BIGSERIAL NOT NULL, 
	CONSTRAINT dim_pollutant_pk PRIMARY KEY (pollutant_id)
)




In [83]:

ddl_pollutant=get_schema(dim_pollutant, name='dim_pollutant', schema='gold',keys=['pollutant_id'],con=engine)
with engine.connect() as conn:
    conn.execute(text(ddl_pollutant))
    conn.commit()

In [84]:
load_table(dim_pollutant,"dim_pollutant",'gold',engine)

load of gold.dim_pollutant done


## Dim_sector

In [99]:
f1_emission.isnull().sum()

reportingyear            0
eprtr_sectorcode         5
eprtr_sectorname         0
facilityinspireid        0
facilityname             0
city                     0
longitude               21
latitude                21
emitted_pollutant_kg     8
pollutant                0
dtype: int64

In [90]:
dim_sector=f1_emission[['eprtr_sectorcode', 'eprtr_sectorname']].drop_duplicates().reset_index(drop=True)

In [95]:
dim_sector.head()

,eprtr_sectorcode,eprtr_sectorname
0,2.0,Production and processing of metals
1,4.0,Chemical industry
2,7.0,Intensive livestock production and aquaculture
3,3.0,Mineral industry
4,1.0,Energy sector


In [97]:
dim_sector[dim_sector['eprtr_sectorcode'].isnull()]

,eprtr_sectorcode,eprtr_sectorname
7,NaN,unknown


In [102]:
dim_sector = dim_sector.dropna()

In [104]:
dim_sector.isnull().sum()

eprtr_sectorcode    0
eprtr_sectorname    0
dtype: int64

In [103]:
ddl_sector = get_schema(dim_sector, name='dim_sector', schema='gold', keys=['eprtr_sectorcode'],con=engine)

with engine.connect() as conn:
    conn.execute(text(ddl_sector))
    conn.commit()

ProgrammingError: (psycopg2.errors.DuplicateTable) relation "dim_sector" already exists

[SQL: 
CREATE TABLE gold.dim_sector (
	eprtr_sectorcode SERIAL NOT NULL, 
	eprtr_sectorname TEXT, 
	CONSTRAINT dim_sector_pk PRIMARY KEY (eprtr_sectorcode)
)

]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [105]:
load_table(dim_sector,'dim_sector','gold',engine)

load of gold.dim_sector done


## Dim_site

In [106]:
f1_emission.head()

,reportingyear,eprtr_sectorcode,eprtr_sectorname,facilityinspireid,facilityname,city,longitude,latitude,emitted_pollutant_kg,pollutant
0,2016,2.0,Production and processing of metals,BE.WA/012010000.FACILITY,INDUSTEEL BELGIUM,Marchienne-Au-Pont,4.411143,50.40992,106000000.0,Carbon dioxide (CO2)
1,2016,4.0,Chemical industry,https://data.ied_registry.omgeving.vlaanderen....,VYNOVA BELGIUM,Tessenderlo-Ham,5.095800,51.05785,1800.0,Hydro-fluorocarbons (HFCS)
2,2021,7.0,Intensive livestock production and aquaculture,https://data.ied_registry.omgeving.vlaanderen....,CONFIDENTIAL -Article4(2)(f),Pittem,3.296810,50.98824,13800.0,Ammonia (NH3)
3,2021,4.0,Chemical industry,https://data.ied_registry.omgeving.vlaanderen....,TESSENDERLO GROUP HAM,Tessenderlo-Ham,5.151300,51.08563,125000.0,Nitrogen oxides (NOX)
4,2016,7.0,Intensive livestock production and aquaculture,https://data.ied_registry.omgeving.vlaanderen....,CONFIDENTIAL -Article4(2)(f),Wevelgem,3.152470,50.82320,12800.0,Ammonia (NH3)


In [113]:
f1_emission.dropna(inplace=True)

In [114]:
f1_emission.isnull().sum()

reportingyear           0
eprtr_sectorcode        0
eprtr_sectorname        0
facilityinspireid       0
facilityname            0
city                    0
longitude               0
latitude                0
emitted_pollutant_kg    0
pollutant               0
dtype: int64

In [139]:
f1_emission.head()

,reportingyear,eprtr_sectorcode,eprtr_sectorname,facilityinspireid,facilityname,city,longitude,latitude,emitted_pollutant_kg,pollutant
0,2016,2.0,Production and processing of metals,BE.WA/012010000.FACILITY,INDUSTEEL BELGIUM,Marchienne-Au-Pont,4.411143,50.40992,106000000.0,Carbon dioxide (CO2)
1,2016,4.0,Chemical industry,https://data.ied_registry.omgeving.vlaanderen....,VYNOVA BELGIUM,Tessenderlo-Ham,5.095800,51.05785,1800.0,Hydro-fluorocarbons (HFCS)
2,2021,7.0,Intensive livestock production and aquaculture,https://data.ied_registry.omgeving.vlaanderen....,CONFIDENTIAL -Article4(2)(f),Pittem,3.296810,50.98824,13800.0,Ammonia (NH3)
3,2021,4.0,Chemical industry,https://data.ied_registry.omgeving.vlaanderen....,TESSENDERLO GROUP HAM,Tessenderlo-Ham,5.151300,51.08563,125000.0,Nitrogen oxides (NOX)
4,2016,7.0,Intensive livestock production and aquaculture,https://data.ied_registry.omgeving.vlaanderen....,CONFIDENTIAL -Article4(2)(f),Wevelgem,3.152470,50.82320,12800.0,Ammonia (NH3)


In [153]:
print(dim_site.shape)
print(dim_site['site_id'].nunique() == dim_site.shape[0])  # doit retourner True

(459, 6)
True


In [129]:
dim_site=f1_emission[['facilityinspireid',	'facilityname',	'city'	,'longitude',	'latitude']]

In [130]:
dim_site = dim_site.drop_duplicates(subset=['facilityinspireid']).reset_index(drop=True)

In [131]:
dim_site['site_id'] = dim_site.index + 1


In [132]:
mapping_site = dim_site.set_index('facilityinspireid')['site_id'].to_dict()

In [134]:
print(dim_site.shape)
print(dim_site['site_id'].nunique() == dim_site.shape[0])  # doit retourner True

(459, 6)
True


In [136]:
ddl_site=get_schema(dim_site, name='dim_site', schema='gold',keys=['site_id'],con=engine)
with engine.connect() as conn:
    conn.execute(text(ddl_site))
    conn.commit()

In [137]:
load_table(dim_site,"dim_site",'gold',engine)

load of gold.dim_site done


## fact_emissions_air

In [138]:
f1_emission.columns

Index(['reportingyear', 'eprtr_sectorcode', 'eprtr_sectorname',
       'facilityinspireid', 'facilityname', 'city', 'longitude', 'latitude',
       'emitted_pollutant_kg', 'pollutant'],
      dtype='str')

In [143]:
f1_emission=f1_emission[['reportingyear', 'eprtr_sectorcode',
       'facilityinspireid','pollutant',
       'emitted_pollutant_kg', ]]

In [160]:
f1_emission['fk_site_id'] = f1_emission['facilityinspireid'].map(mapping_site)
f1_emission['fk_pollutant_id'] = f1_emission['pollutant'].map(mapping_pollutant)

In [161]:
f1_emission.columns

Index(['reportingyear', 'eprtr_sectorcode', 'facilityinspireid', 'pollutant',
       'emitted_pollutant_kg', 'fk_site_id', 'fk_polluant_id',
       'fk_pollutant_id'],
      dtype='str')

In [166]:
fact_emissions_air=f1_emission[['reportingyear', 'eprtr_sectorcode', 'fk_site_id','fk_pollutant_id',
       'emitted_pollutant_kg' ]]

In [167]:
fact_emissions_air.rename(columns={'reportingyear':'year',},inplace=True)

In [171]:
fact_emissions_air

,year,eprtr_sectorcode,fk_site_id,fk_pollutant_id,emitted_pollutant_kg
0,2016,2.0,1,1,106000000.0
1,2016,4.0,2,2,1800.0
2,2021,7.0,3,3,13800.0
3,2021,4.0,4,4,125000.0
4,2016,7.0,5,3,12800.0
...,...,...,...,...,...
4488,2021,2.0,98,7,154000.0
4489,2021,3.0,61,18,737000.0
4490,2022,6.0,19,4,147000.0
4491,2024,5.0,67,1,328000000.0


In [172]:
ddl_fac=get_schema(fact_emissions_air, name='fact__emissions_air', schema='gold',keys=['year','eprtr_sectorcode','fk_site_id','fk_pollutant_id'],con=engine)
with engine.connect() as conn:
    conn.execute(text(ddl_fac))
    conn.commit()

In [175]:
load_table(fact_emissions_air,'fact__emissions_air',schema='gold',engine=engine)

load of gold.fact__emissions_air done


In [ ]:
print(dim_site.shape)
print(dim_site['site_id'].nunique() == dim_site.shape[0])  # doit retourner True

(459, 6)
True
